# Convolutional Vision Transformer (CvT) — B.Tech Classroom Demonstration

### Goal
This notebook implements a **Convolutional Vision Transformer (CvT)** for image classification on **CIFAR-10**.

It is designed to help students understand how CvT combines:

- **Convolution** → local feature extraction and spatial inductive bias
- **Tokenization** → convert image features into a sequence
- **Self-Attention** → learn long-range relationships among image regions
- **MLP / Feed-Forward Network** → transform token features
- **Global pooling + classifier** → predict the image class

> This is a compact teaching implementation inspired by the CvT idea. It is intentionally simpler than research-scale architectures.

### Learning Outcomes
By the end of the notebook, students should be able to:
1. Explain how convolution creates visual tokens.
2. Explain how Q, K and V are generated using convolutional projections.
3. Trace tensor shapes through the model.
4. Train and evaluate a CvT model.
5. Visualize where self-attention is focusing.

## 1. Architecture Used in This Notebook

```text
Input RGB image: 3 × 32 × 32
          │
          ▼
Conv Embedding 1
3 → 64 channels, stride = 2
          │
          ▼
64 × 16 × 16
          │
          ▼
Conv Embedding 2
64 → 128 channels, stride = 2
          │
          ▼
128 × 8 × 8
          │
          ▼
Flatten spatial locations into 64 tokens
Shape: 64 × 128
          │
          ▼
┌───────────────────────────────────┐
│ CvT Transformer Block × 3         │
│                                   │
│ LayerNorm                         │
│ Convolutional Q, K, V projection  │
│ Multi-Head Self-Attention         │
│ Residual connection               │
│ LayerNorm                         │
│ MLP / Feed Forward Network        │
│ Residual connection               │
└───────────────────────────────────┘
          │
          ▼
Mean Pooling over 64 tokens
          │
          ▼
Linear Classifier
          │
          ▼
10 CIFAR-10 Classes
```

### Key difference from a basic ViT
A basic Vision Transformer commonly creates tokens directly from image patches using a linear projection.  
Here, **convolution is used to create the token representation**, and **convolution is also used inside the Q/K/V projections**.

In [1]:
# 2. Imports and reproducibility

import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 3. Load CIFAR-10

CIFAR-10 contains 60,000 colour images of size **32 × 32** belonging to 10 classes.

For a quick classroom demonstration, this notebook uses a subset by default.  
Set `USE_SUBSET = False` for full training.

In [ ]:
# Resource-friendly settings
USE_SUBSET = True
TRAIN_SAMPLES = 5000
TEST_SAMPLES = 1000
BATCH_SIZE = 64

# CIFAR-10 normalization
mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

if USE_SUBSET:
    train_dataset = Subset(train_dataset, range(TRAIN_SAMPLES))
    test_dataset = Subset(test_dataset, range(TEST_SAMPLES))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

  5%|▌         | 9.24M/170M [06:20<2:01:35, 22.1kB/s]

## 4. Display a Few Input Images

In [ ]:
def denormalize(img):
    mean_t = torch.tensor(mean).view(3, 1, 1)
    std_t = torch.tensor(std).view(3, 1, 1)
    return torch.clamp(img.cpu() * std_t + mean_t, 0, 1)

images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 5))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(denormalize(images[i]).permute(1, 2, 0))
    plt.title(classes[labels[i]])
    plt.axis("off")

plt.tight_layout()
plt.show()

print("Input batch shape:", images.shape)
# Expected: [batch_size, 3, 32, 32]

## 5. Convolutional Token Embedding

Instead of cutting the image into patches manually, we use a convolution layer.

For an input feature map

\[
(B, C_{in}, H, W)
\]

the convolution produces

\[
(B, C_{out}, H', W')
\]

Then spatial positions are flattened into tokens:

\[
(B, N, C_{out}), \qquad N = H'W'
\]

Thus, each spatial location becomes one visual token.

In [ ]:
class ConvEmbedding(nn.Module):
    """
    Convolutional token embedding.

    Input:
        [B, C_in, H, W]

    Output:
        tokens [B, N, C_out]
        spatial size H', W'
    """
    def __init__(self, in_channels, embed_dim,
                 kernel_size=3, stride=2, padding=1):
        super().__init__()

        self.proj = nn.Conv2d(
            in_channels,
            embed_dim,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)                 # [B, C, H, W]
        B, C, H, W = x.shape

        x = x.flatten(2).transpose(1, 2) # [B, H*W, C]
        x = self.norm(x)

        return x, H, W

## 6. Convolutional Multi-Head Self-Attention

In ordinary self-attention,

\[
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V
\]

and

\[
Attention(Q,K,V)
=
Softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

### CvT idea used here
Before creating Q, K and V, tokens are reshaped back into a 2-D feature map and passed through:

1. **Depthwise convolution** → learns local spatial patterns
2. **1 × 1 pointwise convolution** → mixes channel information

This introduces convolutional spatial information into self-attention.

In [ ]:
class ConvProjection(nn.Module):
    """Convolutional projection used to generate Q, K, or V."""

    def __init__(self, dim):
        super().__init__()

        self.proj = nn.Sequential(
            # Depthwise convolution
            nn.Conv2d(
                dim, dim,
                kernel_size=3,
                padding=1,
                groups=dim,
                bias=False
            ),

            # Pointwise convolution
            nn.Conv2d(
                dim, dim,
                kernel_size=1,
                bias=False
            )
        )

    def forward(self, x, H, W):
        B, N, C = x.shape

        # Tokens -> 2-D feature map
        x = x.transpose(1, 2).reshape(B, C, H, W)

        # Convolutional projection
        x = self.proj(x)

        # 2-D feature map -> tokens
        x = x.flatten(2).transpose(1, 2)

        return x


class CvTAttention(nn.Module):
    """Multi-head self-attention with convolutional Q/K/V projections."""

    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()

        assert dim % num_heads == 0, \
            "Embedding dimension must be divisible by number of heads."

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = ConvProjection(dim)
        self.k_proj = ConvProjection(dim)
        self.v_proj = ConvProjection(dim)

        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)

        # Used only for classroom visualization
        self.last_attention = None

    def forward(self, x, H, W):
        B, N, C = x.shape

        q = self.q_proj(x, H, W)
        k = self.k_proj(x, H, W)
        v = self.v_proj(x, H, W)

        # [B, N, C] -> [B, heads, N, head_dim]
        q = q.reshape(
            B, N, self.num_heads, self.head_dim
        ).transpose(1, 2)

        k = k.reshape(
            B, N, self.num_heads, self.head_dim
        ).transpose(1, 2)

        v = v.reshape(
            B, N, self.num_heads, self.head_dim
        ).transpose(1, 2)

        # Scaled dot-product attention
        attention_scores = (
            q @ k.transpose(-2, -1)
        ) * self.scale

        attention = attention_scores.softmax(dim=-1)
        attention = self.dropout(attention)

        self.last_attention = attention.detach()

        out = attention @ v

        # [B, heads, N, head_dim] -> [B, N, C]
        out = out.transpose(1, 2).reshape(B, N, C)
        out = self.out_proj(out)

        return out

## 7. Feed-Forward Network (MLP)

After attention, every token is processed independently by:

\[
MLP(x)=W_2(GELU(W_1x))
\]

With `mlp_ratio = 4`, a 128-dimensional token is temporarily expanded to 512 dimensions.

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, dropout=0.1):
        super().__init__()

        hidden_dim = int(dim * mlp_ratio)

        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

## 8. CvT Transformer Block

Each block performs:

\[
x \leftarrow x + Attention(LayerNorm(x))
\]

\[
x \leftarrow x + MLP(LayerNorm(x))
\]

The additions are **residual connections**.

In [ ]:
class CvTBlock(nn.Module):
    def __init__(self, dim, num_heads=4,
                 mlp_ratio=4.0, dropout=0.1):
        super().__init__()

        self.norm1 = nn.LayerNorm(dim)
        self.attn = CvTAttention(
            dim=dim,
            num_heads=num_heads,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(
            dim=dim,
            mlp_ratio=mlp_ratio,
            dropout=dropout
        )

    def forward(self, x, H, W):
        # Attention + residual connection
        x = x + self.attn(self.norm1(x), H, W)

        # MLP + residual connection
        x = x + self.mlp(self.norm2(x))

        return x

## 9. Complete Educational CvT Model

We use two convolutional embedding stages:

- Stage 1: `32 × 32 → 16 × 16`
- Stage 2: `16 × 16 → 8 × 8`
- Final token count: `8 × 8 = 64`

For simplicity, we use **mean pooling** rather than a `[CLS]` token.

In [ ]:
class SimpleCvT(nn.Module):
    def __init__(
        self,
        num_classes=10,
        embed_dim=128,
        num_heads=4,
        depth=3,
        dropout=0.1
    ):
        super().__init__()

        # 3 x 32 x 32 -> 64 x 16 x 16
        self.embed1 = ConvEmbedding(
            in_channels=3,
            embed_dim=64,
            kernel_size=3,
            stride=2,
            padding=1
        )

        # 64 x 16 x 16 -> 128 x 8 x 8
        self.embed2 = ConvEmbedding(
            in_channels=64,
            embed_dim=embed_dim,
            kernel_size=3,
            stride=2,
            padding=1
        )

        self.blocks = nn.ModuleList([
            CvTBlock(
                dim=embed_dim,
                num_heads=num_heads,
                mlp_ratio=4.0,
                dropout=dropout
            )
            for _ in range(depth)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_shapes=False):
        shapes = {}

        # ----- Stage 1 -----
        x, H, W = self.embed1(x)
        shapes["After Conv Embedding 1"] = tuple(x.shape)

        # Tokens -> feature map for the second convolution
        B, N, C = x.shape
        x = x.transpose(1, 2).reshape(B, C, H, W)

        # ----- Stage 2 -----
        x, H, W = self.embed2(x)
        shapes["After Conv Embedding 2"] = tuple(x.shape)

        # ----- CvT blocks -----
        for i, block in enumerate(self.blocks):
            x = block(x, H, W)
            shapes[f"After CvT Block {i+1}"] = tuple(x.shape)

        x = self.norm(x)

        # Mean pooling across visual tokens
        x = x.mean(dim=1)
        shapes["After Mean Pooling"] = tuple(x.shape)

        logits = self.head(x)
        shapes["Logits"] = tuple(logits.shape)

        if return_shapes:
            return logits, shapes

        return logits

## 10. Create the Model and Count Parameters

In [ ]:
model = SimpleCvT(
    num_classes=10,
    embed_dim=128,
    num_heads=4,
    depth=3,
    dropout=0.1
).to(device)

num_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(model)
print(f"\nTotal parameters:     {num_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 11. Trace Tensor Shapes Through the Network

This is one of the most useful cells for teaching.  
Students can see exactly how an image becomes a sequence of visual tokens.

In [ ]:
sample_batch, _ = next(iter(train_loader))
sample_batch = sample_batch[:4].to(device)

model.eval()

with torch.no_grad():
    sample_logits, shape_info = model(
        sample_batch,
        return_shapes=True
    )

print("Input:", tuple(sample_batch.shape))

for name, shape in shape_info.items():
    print(f"{name:25s} -> {shape}")

### Expected shape flow for a batch of 4 images

```text
Input                     -> [4, 3, 32, 32]
After Conv Embedding 1    -> [4, 256, 64]
After Conv Embedding 2    -> [4, 64, 128]
After CvT Block 1         -> [4, 64, 128]
After CvT Block 2         -> [4, 64, 128]
After CvT Block 3         -> [4, 64, 128]
After Mean Pooling        -> [4, 128]
Logits                    -> [4, 10]
```

Interpretation:

- `256 = 16 × 16` tokens after the first convolutional embedding.
- `64 = 8 × 8` tokens after the second convolutional embedding.
- Every final token contains 128 features.

## 12. Define Loss Function and Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

print("Loss     : CrossEntropyLoss")
print("Optimizer: AdamW")

## 13. Training and Evaluation Functions

For a classroom demonstration, start with **2 epochs**.  
The objective is to understand the architecture, not to obtain state-of-the-art accuracy.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total

    return avg_loss, accuracy

## 14. Train the CvT

In [ ]:
EPOCHS = 2

history = {
    "train_loss": [],
    "train_acc": [],
    "test_loss": [],
    "test_acc": []
}

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        device
    )

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.2f}%"
    )

## 15. Plot Training Curves

In [ ]:
epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(6, 4))
plt.plot(
    epochs,
    history["train_loss"],
    marker="o",
    label="Train Loss"
)
plt.plot(
    epochs,
    history["test_loss"],
    marker="o",
    label="Test Loss"
)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CvT Training and Test Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(
    epochs,
    history["train_acc"],
    marker="o",
    label="Train Accuracy"
)
plt.plot(
    epochs,
    history["test_acc"],
    marker="o",
    label="Test Accuracy"
)
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("CvT Training and Test Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 16. Show Predictions on Test Images

In [ ]:
model.eval()

test_images, test_labels = next(iter(test_loader))
test_images_device = test_images.to(device)

with torch.no_grad():
    outputs = model(test_images_device)
    predictions = outputs.argmax(dim=1).cpu()

plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)

    plt.imshow(
        denormalize(test_images[i]).permute(1, 2, 0)
    )

    true_name = classes[test_labels[i]]
    pred_name = classes[predictions[i]]

    plt.title(
        f"True: {true_name}\n"
        f"Pred: {pred_name}"
    )

    plt.axis("off")

plt.tight_layout()
plt.show()

## 17. Visualize Self-Attention

The last CvT block stores an attention matrix of shape:

\[
[B,\ heads,\ N,\ N]
\]

For the final feature map:

- `N = 64`
- spatial arrangement = `8 × 8`

We select the **central query token** and visualize how strongly it attends to the other 63 tokens.

In [ ]:
model.eval()

# Run one image through the network
image = test_images[0:1].to(device)

with torch.no_grad():
    output = model(image)
    predicted_class = output.argmax(dim=1).item()

# Attention from the last transformer block
attention = model.blocks[-1].attn.last_attention

print("Attention tensor shape:", attention.shape)

# Average over heads
attention_mean = attention.mean(dim=1)[0]  # [N, N]

N = attention_mean.shape[0]
side = int(np.sqrt(N))

assert side * side == N, \
    "Token count is not a square number."

# Choose the central token as the query
center_row = side // 2
center_col = side // 2
center_token = center_row * side + center_col

attention_map = attention_mean[
    center_token
].reshape(side, side).cpu()

plt.figure(figsize=(4, 4))
plt.imshow(
    denormalize(test_images[0]).permute(1, 2, 0)
)
plt.title(
    f"Input Image\n"
    f"True: {classes[test_labels[0]]}, "
    f"Pred: {classes[predicted_class]}"
)
plt.axis("off")
plt.show()

plt.figure(figsize=(5, 4))
plt.imshow(
    attention_map,
    interpolation="nearest"
)
plt.colorbar(label="Attention weight")
plt.scatter(
    center_col,
    center_row,
    marker="x",
    s=120
)
plt.title("Attention from the Central Visual Token")
plt.xlabel("Token column")
plt.ylabel("Token row")
plt.show()

## 18. Class-Wise Accuracy

In [ ]:
@torch.no_grad()
def classwise_accuracy(
    model,
    loader,
    device,
    num_classes=10
):
    model.eval()

    correct = torch.zeros(num_classes)
    total = torch.zeros(num_classes)

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        for c in range(num_classes):
            mask = labels == c

            total[c] += mask.sum().cpu()

            correct[c] += (
                (predictions == labels) & mask
            ).sum().cpu()

    return 100.0 * correct / total.clamp(min=1)


acc_per_class = classwise_accuracy(
    model,
    test_loader,
    device
)

for class_name, acc in zip(classes, acc_per_class):
    print(
        f"{class_name:10s}: "
        f"{acc.item():6.2f}%"
    )

# 19. Teaching Summary

### Step 1 — Convolutional embedding
The RGB image is converted into lower-resolution feature maps.  
Each spatial position becomes a visual token.

### Step 2 — Convolutional Q, K and V
Instead of using only linear projections, local spatial information is introduced using depthwise convolutions.

### Step 3 — Multi-head self-attention
Every token can compare itself with every other token.

\[
Attention(Q,K,V)
=
Softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

### Step 4 — Residual connection + MLP
Attention and feed-forward transformations refine the token representations.

### Step 5 — Classification
The final visual tokens are averaged and passed to a linear classifier.

---

## CNN vs ViT vs CvT

| Property | CNN | Basic ViT | CvT |
|---|---|---|---|
| Local spatial bias | Strong | Weak initially | Stronger than ViT |
| Long-range interaction | Indirect | Strong | Strong |
| Token generation | Convolution features | Image patches | Convolutional embedding |
| Q/K/V projection | N/A | Linear | Convolutional |
| Positional information | Strong locality | Usually positional embedding | Partly supplied by convolution |

# 20. Suggested B.Tech Lab Exercises

1. Change the number of attention heads from `4` to `8`.
2. Change the number of CvT blocks from `3` to `1`, then to `4`.
3. Increase `TRAIN_SAMPLES` and observe accuracy.
4. Train for `10` epochs instead of `2`.
5. Remove the depthwise convolution from `ConvProjection` and compare results.
6. Replace mean pooling with a learnable `[CLS]` token.
7. Compare this CvT with a small CNN and a basic ViT.
8. Print the shapes of Q, K and V inside the attention module.
9. Visualize attention for different query tokens.
10. Explain why the attention matrix has shape `N × N`.

### Viva Questions

- Why is convolution useful before self-attention?
- Why is the factor \(1/\sqrt{d_k}\) used?
- Why are multiple attention heads used?
- What is the role of residual connections?
- Why can CvT preserve local spatial structure better than a patch-only ViT?

# 21. Optional Challenge: Inspect Q, K and V Shapes

Inside `CvTAttention.forward()`, students can temporarily add:

```python
print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)
```

The expected shape is:

```text
[B, number_of_heads, number_of_tokens, head_dimension]
```

For this notebook:

- embedding dimension = 128
- heads = 4
- head dimension = 128 / 4 = 32
- final tokens = 64

Hence:

```text
[B, 4, 64, 32]
```